When a face can't be recognized by the landmark recognizer 
We make all 32 geometry feature = 0
So the prediction will completely rely on the 512 features of ResNet18


there are 7 normal images
with one image that has no face: 000199
and we'll test on this sample later

DataLoader returns:
small_images
small_labels
small_points
filenames
small_valid

small_valid of the 8 images is [True, True, True, True, True, True, True, False] 
which means the last image doens't have a landmark
this is called a mask

valid=False 的真正作用就是：告诉模型“这张图片的 geometry 不可信
把这张图的 32 个 geometry features 全部关掉

不要把它理解成“把整个 landmark FC 层变成 0”。不是的。FC/MLP 本身还保留、还正常训练
只是针对这一张 valid=False 的图片，把它最终产生的 geometry feature 清零。

normally a landmark has ten features: [0.3, 0.8, 0.4, ... 0.2]
after MLP, it becomes a 32 tensor
valid=False makes : 32 geometry_features × 0 = [0,0,0,...,0]

valid 会作为第三个输入传给 M3： model(images, points, valid)

In geometry MLP:
the first layer is : 10 -> 32: nn.Linear(10, 32)
and it will experience: y = Wx+b
if we simply does: no_face → landmarks = [0,0,...,0]
then x = 0
y=W(0)+b= b !!!!!
so at the end of the day, we need:  valid = False
so we can get [512  image features | 32  0]


In [ ]:
from pathlib import Path
import subprocess

import torch
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/FacialVisualProfile")
workspace = Path("/content/day11_geometry_workspace")
expected_commit = "a6722d6"

if not workspace.exists():
    subprocess.run(
        [
            "git", "clone", "--no-checkout",
            "https://github.com/DeweyYip/facial-visual-profile.git",
            str(workspace),
        ],
        check=True,
    )
    subprocess.run(
        [
            "git", "-C", str(workspace),
            "checkout", "--detach", expected_commit,
        ],
        check=True,
    )
else:
    status = subprocess.check_output(
        ["git", "-C", str(workspace), "status", "--porcelain"],
        text=True,
    )
    if status.strip():
        raise RuntimeError("Day 11 workspace has local changes.")

commit = subprocess.check_output(
    ["git", "-C", str(workspace), "rev-parse", "HEAD"],
    text=True,
).strip()
assert commit.startswith(expected_commit), commit

print("Workspace:", workspace)
print("Source commit:", commit)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n=== Baseline source snapshots ===")
for run_name in ("r18_seed0", "r50_seed0"):
    run_dir = drive_root / "runs" / run_name
    source_dir = run_dir / "source"

    print(f"\n{run_name}:")
    if not source_dir.is_dir():
        print("Source directory not found:", source_dir)
        continue

    for path in sorted(source_dir.rglob("*")):
        if path.is_file():
            print(path.relative_to(source_dir))

    for name in ("config.json", "best.json"):
        path = run_dir / name
        print(f"{name}: {'FOUND' if path.is_file() else 'MISSING'}")

print("\n=== Required data ===")
for path in [
    drive_root / "cache/mtcnn_clean.sqlite",
    drive_root / "data/img_align_celeba.zip",
    drive_root / "data/list_attr_celeba.txt",
    drive_root / "data/list_landmarks_align_celeba.txt",
    drive_root / "data/list_eval_partition.txt",
]:
    if not path.is_file():
        raise FileNotFoundError(path)
    print(path.name, "FOUND")

print("\nDay 11 workspace setup: PASS")

In [ ]:
import difflib
import hashlib
import json

shared_files = [
    "configs/attributes.yaml",
    "datasets/preprocessing.py",
    "datasets/celeba.py",
    "evaluation/attributes.py",
]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

comparison_results = []

for name in shared_files:
    current = workspace / name
    r18 = drive_root / "runs/r18_seed0/source" / name
    r50 = drive_root / "runs/r50_seed0/source" / name

    for path in (current, r18, r50):
        if not path.is_file():
            raise FileNotFoundError(path)

    print(f"\n=== {name} ===")
    print("R18:", sha256(r18))
    print("R50:", sha256(r50))
    print("Current:", sha256(current))

    baseline_equal = r18.read_bytes() == r50.read_bytes()
    current_equal = current.read_bytes() == r18.read_bytes()

    print("R18 == R50:", baseline_equal)
    print("Current == R18:", current_equal)

    comparison_results.append({
        "file": name,
        "r18_sha256": sha256(r18),
        "r50_sha256": sha256(r50),
        "current_sha256": sha256(current),
        "baselines_equal": baseline_equal,
        "current_equals_r18": current_equal,
    })

    for label, other in [
        ("R50", r50),
        ("Current", current),
    ]:
        if other.read_bytes() != r18.read_bytes():
            print(f"\n--- R18 vs {label} ---")
            difference = list(difflib.unified_diff(
                r18.read_text(encoding="utf-8").splitlines(),
                other.read_text(encoding="utf-8").splitlines(),
                fromfile=f"R18/{name}",
                tofile=f"{label}/{name}",
                lineterm="",
            ))
            print("\n".join(difference[:200]))
            if len(difference) > 200:
                print(f"... {len(difference) - 200} more diff lines")

print("\n=== Saved baseline configurations ===")
for run_name in ("r18_seed0", "r50_seed0"):
    path = drive_root / "runs" / run_name / "config.json"
    print(f"\n{run_name}:")
    print(json.dumps(
        json.loads(path.read_text(encoding="utf-8")),
        indent=2,
    ))

print("\nSource comparison complete; differences require inspection.")

In [ ]:
import ast
import json
import yaml


class RemoveDocumentation(ast.NodeTransformer):
    def visit_Expr(self, node):
        if isinstance(node.value, ast.Constant):
            if isinstance(node.value.value, str):
                return None
        return self.generic_visit(node)


def computational_ast(path):
    tree = ast.parse(path.read_text(encoding="utf-8"))
    tree = RemoveDocumentation().visit(tree)
    return ast.dump(tree, include_attributes=False)


configs = {}
for run_name in ("r18_seed0", "r50_seed0"):
    run_dir = drive_root / "runs" / run_name
    config = json.loads(
        (run_dir / "config.json").read_text(encoding="utf-8")
    )
    configs[run_name] = config

    # Verify each saved source against its recorded hash.
    for name, expected_hash in config["source_sha256"].items():
        assert sha256(run_dir / "source" / name) == expected_hash, (
            run_name, name
        )
    print(f"{run_name}: saved source hashes PASS")

for result in comparison_results:
    name = result["file"]
    current = workspace / name
    r18 = drive_root / "runs/r18_seed0/source" / name
    r50 = drive_root / "runs/r50_seed0/source" / name

    # Ensure the files have not changed since the previous comparison.
    assert sha256(current) == result["current_sha256"]
    assert sha256(r18) == result["r18_sha256"]
    assert sha256(r50) == result["r50_sha256"]

    if name.endswith(".py"):
        assert (
            computational_ast(current)
            == computational_ast(r18)
            == computational_ast(r50)
        ), name
    else:
        assert current.read_bytes() == r18.read_bytes() == r50.read_bytes()

    result["computational_equivalence"] = True
    print(f"{name}: computational equivalence PASS")

with (workspace / "configs/attributes.yaml").open(encoding="utf-8") as f:
    attribute_names = yaml.safe_load(f)["attributes"]

shared_settings = [
    "seed", "epochs", "batch_size", "num_workers",
    "optimizer", "learning_rate", "weight_decay", "precision",
    "augmentation", "f1_threshold", "selection_metric",
    "epoch_seed_rule", "cudnn_deterministic", "cudnn_benchmark",
]

for key in shared_settings:
    assert configs["r18_seed0"][key] == configs["r50_seed0"][key], key

for config in configs.values():
    assert config["attribute_names"] == attribute_names

report = {
    "source_commit": commit,
    "files": comparison_results,
    "saved_baseline_source_hashes_verified": True,
    "attribute_order_verified": True,
    "shared_baseline_settings": {
        key: configs["r18_seed0"][key] for key in shared_settings
    },
    "scope": (
        "Shared source computation and baseline configurations checked; "
        "M3 training and GPU environment have not yet been checked."
    ),
}

report_path = (
    drive_root / "runs/m3_preparation/day11/source_compatibility.json"
)
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(
    json.dumps(report, indent=2) + "\n",
    encoding="utf-8",
)

print("Saved:", report_path)
print("Day 11 baseline source compatibility: PASS")

This the place to test if failure_policy="mask" works

In [ ]:
from pathlib import Path
import hashlib
import importlib
import json
import os
import random
import shutil
import sys
import zipfile

import torch
from torch.utils.data import DataLoader, Subset

assert torch.cuda.is_available(), "Select a GPU runtime first."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

workspace = Path("/content/day11_geometry_workspace")
drive_root = Path("/content/drive/MyDrive/FacialVisualProfile")
local_dir = Path("/content/day11_geometry_data")
raw_root = local_dir / "raw"
image_dir = raw_root / "img_align_celeba"
image_dir.mkdir(parents=True, exist_ok=True)

report_path = (
    drive_root / "runs/m3_preparation/day11/source_compatibility.json"
)
compatibility = json.loads(report_path.read_text(encoding="utf-8"))
assert compatibility["source_commit"] == commit

for record in compatibility["files"]:
    actual_hash = hashlib.sha256(
        (workspace / record["file"]).read_bytes()
    ).hexdigest()
    assert actual_hash == record["current_sha256"]
    assert record["computational_equivalence"]

# Use only this workspace's datasets package.
for name, module in list(sys.modules.items()):
    if name == "datasets" or name.startswith("datasets."):
        origin = getattr(module, "__file__", None)
        if origin is None or not Path(origin).resolve().is_relative_to(
            workspace.resolve()
        ):
            raise RuntimeError("Another datasets package is already imported.")

sys.path.insert(0, str(workspace))
importlib.invalidate_caches()
os.chdir(workspace)

from datasets.celeba import CelebAAttributes
from datasets.celeba_predicted import CelebAPredictedAttributes
from datasets.landmark_cache import MTCNNLandmarkCache

for name in [
    "list_attr_celeba.txt",
    "list_landmarks_align_celeba.txt",
    "list_eval_partition.txt",
]:
    shutil.copy2(drive_root / "data" / name, raw_root / name)

cache_path = local_dir / "mtcnn_clean.sqlite"
shutil.copy2(drive_root / "cache/mtcnn_clean.sqlite", cache_path)

cache = MTCNNLandmarkCache(
    cache_path, raw_root / "list_eval_partition.txt", split="train"
)

successful_names = sorted(
    name for name, encoded in cache.records.items()
    if json.loads(encoded)["status"] == "ok"
)

"""
there are 7 normal images
with one image that has no face: 000199
and we'll test on this sample later

DataLoader returns:
small_images
small_labels
small_points
filenames
small_valid

small_valid of the 8 images is [True, True, True, True, True, True, True, False] 
which means the last image doens't have a landmark
this is called a mask

valid=False 的真正作用就是：告诉模型“这张图片的 geometry 不可信
把这张图的 32 个 geometry features 全部关掉

不要把它理解成“把整个 landmark FC 层变成 0”。不是的。FC/MLP 本身还保留、还正常训练
只是针对这一张 valid=False 的图片，把它最终产生的 geometry feature 清零。

normally a landmark has ten features: [0.3, 0.8, 0.4, ... 0.2]
after MLP, it becomes a 32 tensor
valid=False makes : 32 geometry_features × 0 = [0,0,0,...,0]

valid 会作为第三个输入传给 M3： model(images, points, valid)

In geometry MLP:
the first layer is : 10 -> 32: nn.Linear(10, 32)
and it will experience: y = Wx+b
if we simply does: no_face → landmarks = [0,0,...,0]
then x = 0
y=W(0)+b= b !!!!!
so at the end of the day, we need:  valid = False
so we can get [512  image features | 32  0]

"""
sample_names = random.Random(0).sample(successful_names, 7)
sample_names.append("000199.jpg")

assert json.loads(cache.records["000199.jpg"])["status"] == "no_face"

manifest = []
with zipfile.ZipFile(drive_root / "data/img_align_celeba.zip") as archive:
    needed = set(sample_names)
    members = {}
    for member in archive.namelist():
        if member.endswith("/"):
            continue
        name = Path(member).name
        if name in needed:
            if name in members:
                raise ValueError(f"Duplicate image filename: {name}")
            members[name] = member

    assert set(members) == needed
    for name in sample_names:
        content = archive.read(members[name])
        record = json.loads(cache.records[name])
        image_hash = hashlib.sha256(content).hexdigest()
        assert image_hash == record["image_sha256"]
        (image_dir / name).write_bytes(content)
        manifest.append({
            "name": name,
            "status": record["status"],
            "image_sha256": image_hash,
        })

del cache

"""
This is the most crucial part
it sets failure_policy = "mask"
it means:
if our model encounters no_face
Do not raise error, instead, use mask
"""
train_dataset = CelebAPredictedAttributes(
    cache_path,
    split="train",
    root=raw_root,
    failure_policy="mask",
)
baseline = CelebAAttributes(split="train", root=raw_root)

sample_indices = [
    train_dataset.names.index(name) for name in sample_names
]
for index in sample_indices:
    sample = train_dataset[index]
    reference = baseline[index]
    assert sample[3] == reference[3]
    assert torch.equal(sample[0], reference[0])
    assert torch.equal(sample[1], reference[1])
del baseline

small_dataset = Subset(train_dataset, sample_indices)
small_loader = DataLoader(
    small_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
)

small_images, small_labels, small_points, filenames, small_valid = (
    next(iter(small_loader))
)

assert small_images.shape == (8, 3, 224, 224)
assert small_labels.shape == (8, 24)
assert small_points.shape == (8, 10)
assert small_valid.dtype == torch.bool
assert small_valid.tolist() == [True] * 7 + [False]
assert list(filenames) == sample_names

# the 8th picture's landmark tensor is [0,0,0,0,0,0,0,0,0,0]
assert torch.equal(small_points[-1], torch.zeros(10))

assert torch.isfinite(small_images).all()
assert torch.isfinite(small_points).all()
assert ((small_labels == 0) | (small_labels == 1)).all()

output_dir = drive_root / "runs/m3_preparation/day11"
(output_dir / "overfit_samples.json").write_text(
    json.dumps({
        "source_commit": commit,
        "sampling_seed": 0,
        "split": "train",
        "samples": manifest,
        "geometry_valid": small_valid.tolist(),
        "dataloader_workers": 2,
        "baseline_image_and_label_equality": "PASS",
    }, indent=2) + "\n",
    encoding="utf-8",
)

print("Sample filenames:", list(filenames))
print("Geometry valid:", small_valid.tolist())
print("Images:", tuple(small_images.shape))
print("Labels:", tuple(small_labels.shape))
print("Points:", tuple(small_points.shape))
print("Saved:", output_dir / "overfit_samples.json")
print("Day 11 real mixed-batch preparation: PASS")

We use the images prepared in block 4 to actually out them in the M3 model
Namely, an 8-image overfit smoke test


In [ ]:
import hashlib
import json
import random
import time
from importlib import metadata

import numpy as np
import torch
from torch import nn

from models.resnet18_geometry import ResNet18GeometryAttributes

seed = 0
max_steps = 200
learning_rate = 0.001
loss_target = 0.02

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

assert torch.cuda.is_available()
assert small_valid.tolist() == [True] * 7 + [False]

device = torch.device("cuda")
images = small_images.to(device)
labels = small_labels.to(device)
points = small_points.to(device)
valid = small_valid.to(device)

model = ResNet18GeometryAttributes(pretrained=True).to(device)
assert all(p.requires_grad for p in model.parameters())

optimizer = torch.optim.Adam(
    model.parameters(), lr=learning_rate, weight_decay=0.0
)
criterion = nn.BCEWithLogitsLoss()

output_dir = drive_root / "runs/m3_preparation/day11"
source_dir = output_dir / "source"
source_dir.mkdir(parents=True, exist_ok=True)

source_hashes = {}
for name in [
    "configs/attributes.yaml",
    "datasets/celeba.py",
    "datasets/preprocessing.py",
    "datasets/landmark_cache.py",
    "datasets/celeba_predicted.py",
    "evaluation/attributes.py",
    "models/resnet18_geometry.py",
]:
    source = workspace / name
    destination = source_dir / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    source_hashes[name] = hashlib.sha256(source.read_bytes()).hexdigest()

history = []

@torch.inference_mode()
def evaluate_small_batch(step):
    model.eval()
    logits = model(images, points, valid)
    assert torch.isfinite(logits).all()

    loss = criterion(logits, labels)
    assert torch.isfinite(loss)

    correct = (logits >= 0) == labels.bool()
    exact = correct.all(dim=1)

    result = {
        "step": step,
        "eval_bce": loss.item(),
        "attribute_accuracy": correct.float().mean().item(),
        "exact_match": exact.float().mean().item(),
        "failed_sample_exact": bool(exact[-1].item()),
    }
    history.append(result)
    print(
        f"step={step:3d}  BCE={result['eval_bce']:.6f}  "
        f"attribute_acc={result['attribute_accuracy']:.2%}  "
        f"exact_match={result['exact_match']:.2%}  "
        f"failed_sample_exact={result['failed_sample_exact']}",
        flush=True,
    )
    return result

captured = {}
def capture_geometry(module, inputs):
    captured["geometry"] = inputs[0][:, 512:].detach()

handle = model.classifier.register_forward_pre_hook(capture_geometry)

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.perf_counter()

evaluate_small_batch(0)
gradient_checks = {}
passed = False

for step in range(1, max_steps + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    logits = model(images, points, valid)
    loss = criterion(logits, labels)
    assert torch.isfinite(loss)
    loss.backward()

    if step == 1:
        assert torch.count_nonzero(
            captured["geometry"][-1]
        ).item() == 0

        for name, module in [
            ("image_encoder", model.image_encoder),
            ("geometry_encoder", model.geometry_encoder),
            ("classifier", model.classifier),
        ]:
            parameters = [
                p for p in module.parameters() if p.requires_grad
            ]
            assert all(p.grad is not None for p in parameters)
            assert all(torch.isfinite(p.grad).all() for p in parameters)

            total = sum(
                p.grad.abs().sum().item() for p in parameters
            )
            assert total > 0, name
            gradient_checks[name] = total

        handle.remove()
        print("Real mixed-batch features and gradients: PASS", flush=True)

    optimizer.step()

    if step == 1 or step % 10 == 0:
        result = evaluate_small_batch(step)
        if (
            result["exact_match"] == 1.0
            and result["eval_bce"] <= loss_target
        ):
            passed = True
            break

torch.cuda.synchronize()
elapsed = time.perf_counter() - started
final = history[-1]

report = {
    "result": "PASS" if passed else "NEEDS_INVESTIGATION",
    "experiment_type": "eight-training-image overfit smoke check",
    "source_commit": commit,
    "source_sha256": source_hashes,
    "seed": seed,
    "sample_names": list(filenames),
    "geometry_valid": small_valid.tolist(),
    "initial_weights": "ResNet18_Weights.IMAGENET1K_V1",
    "optimizer": "Adam",
    "learning_rate": learning_rate,
    "weight_decay": 0.0,
    "precision": "FP32",
    "augmentation": "None",
    "max_steps": max_steps,
    "completed_steps": step,
    "pass_criteria": {"exact_match": 1.0, "eval_bce_max": loss_target},
    "first_backward_gradient_sums": gradient_checks,
    "final": final,
    "history": history,
    "elapsed_seconds": elapsed,
    "peak_allocated_gpu_gib": (
        torch.cuda.max_memory_allocated() / 1024**3
    ),
    "environment": {
        "torch": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        **{
            name: metadata.version(name)
            for name in ("torchvision", "numpy", "Pillow", "scikit-learn")
        },
    },
    "scope": (
        "Training-pipeline check on the same eight training images; "
        "not a generalization or baseline-comparison result."
    ),
}

report_path = output_dir / "overfit_report.json"
report_path.write_text(
    json.dumps(report, indent=2) + "\n", encoding="utf-8"
)

checkpoint_path = output_dir / "overfit_smoke.pt"
torch.save({
    "experiment_type": "overfit_smoke_only",
    "source_commit": commit,
    "model_state_dict": {
        name: tensor.detach().cpu()
        for name, tensor in model.state_dict().items()
    },
    "completed_steps": step,
    "sample_names": list(filenames),
    "report_result": report["result"],
}, checkpoint_path)

print(f"Elapsed: {elapsed:.1f} seconds")
print("Saved:", report_path)
print("Saved:", checkpoint_path)
print("Day 11 overfit check:", report["result"])

In [ ]:
from pathlib import Path
import json
import zipfile

from google.colab import files

output_dir = drive_root / "runs/m3_preparation/day11"
report = json.loads(
    (output_dir / "overfit_report.json").read_text(encoding="utf-8")
)
assert report["result"] == "PASS"
assert report["source_commit"] == commit

report_names = [
    "source_compatibility.json",
    "overfit_samples.json",
    "overfit_report.json",
]

archive_path = Path("/content/day11_results.zip")
with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for name in report_names:
        path = output_dir / name
        if not path.is_file():
            raise FileNotFoundError(path)
        archive.write(path, arcname=name)

print("Report archive:", archive_path)
files.download(str(archive_path))